In [11]:
import os

base_dir = os.getcwd()
caminho = os.path.join(base_dir, 'instances', 'uf20-01.cnf')

In [ ]:
# def parser_dimacs(caminho_arquivo):
#     clausulas = []
#     n_variaveis = 0
#     n_clausulas = 0

#     with open(caminho_arquivo, 'r') as arquivo:
#         for linha in arquivo:
#             # Remove espaços em branco nas extremidades
#             linha = linha.strip()
            
#             # 1. Ignora linhas vazias ou comentários
#             if not linha or linha.startswith('c'):
#                 continue
                
#             # 2. Lê o cabeçalho (p cnf n_vars n_claus)
#             if linha.startswith('p'):
#                 partes = linha.split()
#                 n_variaveis = int(partes[2])
#                 n_clausulas = int(partes[3])
#                 continue
            
#             # 3. Lê as cláusulas
#             # Converte a linha em números inteiros
#             valores = list(map(int, linha.split()))
            
#             # O '0' no final serve apenas como delimitador de fim de cláusula[cite: 1]
#             # Removemos o zero antes de armazenar na memória
#             if valores[-1] == 0:
#                 valores.pop()
            
#             clausulas.append(valores)

#     return n_variaveis, n_clausulas, clausulas
def parser_dimacs(caminho_arquivo):
    n_vars = 0
    n_clausulas_esperadas = 0
    lista_clausulas = []
    clausula_atual = []

    try:
        with open(caminho_arquivo, 'r') as f:
            for linha in f:
                linha = linha.strip()
                
                # 1. Ignorar comentários e marcadores de fim de arquivo
                if not linha or linha.startswith(('c', '%', 'w')):
                    continue
                
                # 2. Ler o cabeçalho (p cnf n_vars n_clausulas)
                if linha.startswith('p'):
                    partes = linha.split()
                    n_vars = int(partes[2])
                    n_clausulas_esperadas = int(partes[3])
                    continue
                
                # 3. Processar literais
                # Convertemos todos os itens da linha para inteiros
                valores = list(map(int, linha.split()))
                
                for v in valores:
                    if v == 0:
                        # Encontrou o delimitador, fecha a cláusula atual
                        lista_clausulas.append(clausula_atual)
                        clausula_atual = []
                    else:
                        # Adiciona o literal à cláusula em construção
                        clausula_atual.append(v)
                        
        return n_vars, n_clausulas_esperadas, lista_clausulas

    except FileNotFoundError:
        print(f"Erro: Arquivo '{caminho_arquivo}' não encontrado.")
        return None
    except Exception as e:
        print(f"Erro ao processar o arquivo: {e}")
        return None

In [ ]:
n_vars, n_claus, lista_clausulas = parser_dimacs(caminho)
# n_vars, n_claus, lista_clausulas = parser_dimacs('./instances/uf20-01.cnf')
print(f"Variáveis: {n_vars}, Cláusulas: {n_claus}")
print(f"Primeira cláusula: {lista_clausulas[0]}")
print(f"Segunda cláusula: {lista_clausulas[1]}")
print(f"Terceira cláusula: {lista_clausulas[2]}")

ValueError: invalid literal for int() with base 10: '%'

In [ ]:
def calcular_fo(solucao, clausulas):
    """
    Calcula a quantidade de cláusulas satisfeitas (Maximização).
   
    """
    satisfeitas = 0
    
    for clausula in clausulas:
        # Uma cláusula 3-SAT é uma disjunção (OU)
        clausula_valida = False
        
        for literal in clausula:
            # Variável no arquivo começa em 1, índice no Python começa em 0
            indice = abs(literal) - 1
            valor_variavel = solucao[indice]
            
            if literal > 0:
                # Se o literal é positivo, ele é satisfeito se a variável for 1 (True)
                if valor_variavel == 1:
                    clausula_valida = True
                    break # Curto-circuito: se um literal é True, a cláusula 'OU' é True
            else:
                # Se o literal é negativo, ele é satisfeito se a variável for 0 (False)
                if valor_variavel == 0:
                    clausula_valida = True
                    break
        
        if clausula_valida:
            satisfeitas += 1
            
    return satisfeitas

In [ ]:
# import random

# def gerar_solucao_inicial(n_vars):
#     return [random.randint(0, 1) for _ in range(n_vars)]
import random

# 1. Gerar solução aleatória
def gerar_solucao_inicial(n_vars):
    return [random.randint(0, 1) for _ in range(n_vars)]

# 2. Simulação do Teste
if __name__ == "__main__":
    # Supondo que você já carregou os dados do arquivo uf20-01.cnf
    # n_vars, n_claus, lista_clausulas = parser_dimacs('uf20-01.cnf')
    
    # Exemplo manual para teste rápido:
    n_vars = 20
    # Exemplo de cláusulas: (x1 ou x2 ou !x3), (!x1 ou x4 ou x5)
    exemplo_clausulas = [[1, 2, -3], [-1, 4, 5]] 
    
    solucao_teste = gerar_solucao_inicial(n_vars)
    resultado = calcular_fo(solucao_teste, exemplo_clausulas)
    
    print(f"Vetor Solução Aleatório: {solucao_teste}")
    print(f"Cláusulas Satisfeitas: {resultado} de {len(exemplo_clausulas)}")

Vetor Solução Aleatório: [0, 1, 0, 0, 1, 1, 0, 1, 1, 1, 1, 0, 0, 1, 1, 1, 1, 1, 1, 0]
Cláusulas Satisfeitas: 2 de 2


In [ ]:
#Rotina de Vizinhança
import random
import math

def gerar_vizinho(solucao_atual, percentual=0.05):
    """
    Realiza bit-flip em uma porcentagem das variáveis.
    """
    nova_solucao = list(solucao_atual)
    n_bits = len(nova_solucao)
    
    # Identifica a quantidade de bits a serem modificados (mínimo 1)
    qtd_modificar = max(1, int(n_bits * percentual))
    
    # Escolhe de maneira aleatória os índices que sofrerão modificação
    indices = random.sample(range(n_bits), qtd_modificar)
    
    for idx in indices:
        # Inverte o valor (0 -> 1 ou 1 -> 0)
        nova_solucao[idx] = 1 - nova_solucao[idx]
        
    return nova_solucao

In [ ]:
#Algoritmo Simulated Annealing
def simulated_annealing_3sat(clausulas, n_vars, T0, alpha, SAmax, EvalMax):
    # Inicialização conforme pseudo-código
    s_atual = [random.randint(0, 1) for _ in range(n_vars)]
    fo_atual = calcular_fo(s_atual, clausulas)
    
    s_best = list(s_atual)
    fo_best = fo_atual
    
    T = T0
    avaliacoes = 1 # Contador para o critério de parada
    historico_fo = [] # Dados para o gráfico
    
    # Loop principal (Critério de parada: avaliações ou temperatura)
    while avaliacoes < EvalMax and T > 0.0001:
        iter_t = 0
        
        while iter_t < SAmax and avaliacoes < EvalMax:
            iter_t += 1
            
            # Geração de vizinho e cálculo de delta
            s_prime = gerar_vizinho(s_atual)
            fo_prime = calcular_fo(s_prime, clausulas)
            avaliacoes += 1
            
            # Como é MAXIMIZAÇÃO, delta = fo_atual - fo_prime (piora se delta > 0)
            # Ou mantemos a lógica do pseudo-código adaptando para max:
            delta = fo_atual - fo_prime # Se fo_prime > fo_atual, delta é negativo (melhora)
            
            if delta < 0: # Melhora encontrada
                s_atual = list(s_prime)
                fo_atual = fo_prime
                if fo_prime > fo_best: # Atualiza global
                    s_best = list(s_prime)
                    fo_best = fo_prime
            else:
                # Critério de Metropolis para aceitar piora
                x = random.uniform(0, 1)
                if x < math.exp(-delta / T):
                    s_atual = list(s_prime)
                    fo_atual = fo_prime
            
            # Salva o progresso para o gráfico
            historico_fo.append(fo_atual)
            
        # Queda de temperatura (Equação não-linear/geométrica)
        T = T * alpha
        
    return s_best, fo_best, historico_fo

In [ ]:
#Geração do gráfico de convergência
import matplotlib.pyplot as plt

def plotar_convergencia(historico):
    plt.figure(figsize=(10, 6))
    plt.plot(historico, color='blue', linewidth=1)
    plt.title('Gráfico de Convergência - Simulated Annealing 3-SAT')
    plt.xlabel('Número de Avaliações (Iterações)')
    plt.ylabel('Valor da Função Objetivo (Cláusulas Satisfeitas)')
    plt.grid(True)
    plt.show()

In [ ]:
# %%
# Bloco de Execução Final


# Função objetivo do problema:
# quantidade de cláusulas satisfeitas. Maximizar cláusulas satisfeitas (maximização).

# Explicação:

# T0 = 100.0: Começamos com calor suficiente para que o algoritmo aceite quase qualquer mudança, 
# explorando todo o mapa de soluções.

# alpha = 0.99: Um resfriamento muito gradual. 
# Se  usar 0.90, o sistema "congela" rápido demais e pode ficar preso em um resultado ruim.

# SAmax = 100: É o tempo que o algoritmo passa "estabilizando" em uma temperatura antes de esfriar um pouco mais.

# EvalMax = 200000: Garante que o programa não rode para sempre, parando ao atingir o esforço c
# omputacional definido no  projeto.

if __name__ == "__main__":
    # 1. Configuração do caminho do arquivo
    base_dir = os.getcwd()
    caminho_arquivo = os.path.join(base_dir, 'instances', 'uf20-01.cnf')

    # 2. Carregar a instância
    n_vars, n_claus, lista_clausulas = parser_dimacs(caminho_arquivo)
    print(f"Instância carregada: {n_vars} variáveis e {n_claus} cláusulas.")

    # 3. Definição dos parâmetros do Simulated Annealing
    T0 = 100.0        # Temperatura Inicial
    alpha = 0.99      # Taxa de decaimento (resfriamento lento)
    SAmax = 100       # Iterações para equilíbrio térmico em cada T
    EvalMax = 200000  # Critério de parada: máximo de avaliações

    # 4. Executar o Algoritmo
    print("Executando Simulated Annealing...")
    melhor_sol, melhor_fo, historico = simulated_annealing_3sat(
        lista_clausulas, n_vars, T0, alpha, SAmax, EvalMax
    )

    # 5. Resultados Finais
    print("-" * 30)
    print(f"Melhor resultado encontrado (FO): {melhor_fo} de {n_claus}")
    print(f"Satisfatibilidade: {(melhor_fo/n_claus)*100:.2f}%")
    
    # 6. Gerar Gráfico de Convergência
    plotar_convergencia(historico)

ValueError: invalid literal for int() with base 10: '%'